In [ ]:
{
  "cells": [
    {
      "cell_type": "markdown",
      "source": [
        "# Autonomous IT Support Agent: Core Logic"
      ],
      "metadata": {
        "id": "e17e6bc8"
      }
    },
    {
      "cell_type": "markdown",
      "source": [
        "This section contains the complete, consolidated Python code for the Autonomous IT Support Agent, including all necessary imports, tool definitions, the agent schema, and the main execution loop. This is the 'agent info' you requested."
      ],
      "metadata": {
        "id": "44c4e378"
      }
    },
    {
      "cell_type": "code",
      "source": [
        "import os\n",
        "import json\n",
        "import random # Needed for escalate_to_engineer\n",
        "from openai import OpenAI\n",
        "from google.colab import userdata\n\n",
        "# Initialize OpenAI client\n",
        "client = OpenAI(api_key=userdata.get('OPENAI_APIKEY'))\n\n",
        "# --- Tool 1: Check Health ---\n",
        "def get_server_health(server_id: str) -> str:\n",
        "    \"\"\"Returns CPU and Memory usage for a given server.\"\"\"\n",
        "    print(f\"-> TOOL: Checking health for {server_id}...\")\n\n",
        "    metrics = {\n",
        "        \"payment-server-01\": {\"cpu\": \"98%\", \"memory\": \"40%\", \"status\": \"Warning\"},\n",
        "        \"db-node-02\": {\"cpu\": \"12%\", \"memory\": \"60%\", \"status\": \"Healthy\"},\n",
        "        \"auth-service-03\": {\"cpu\": \"45%\", \"memory\": \"95%\", \"status\": \"Critical\"},\n",
        "        \"search-index-09\": {\"cpu\": \"10%\", \"memory\": \"15%\", \"status\": \"Error\"},\n",
        "        \"frontend-node-04\": {\"cpu\": \"25%\", \"memory\": \"30%\", \"status\": \"Healthy\"},\n",
        "    }\n\n",
        "    result = metrics.get(server_id, {\"error\": \"Server not found. Check the ID.\"})\n",
        "    return json.dumps(result)\n\n",
        "# --- Tool 2: Fetch Recent Logs ---\n",
        "def fetch_recent_logs(server_id: str, lines: int = 5) -> str:\n",
        "    \"\"\"Returns the last N lines of logs.\"\"\"\n",
        "    print(f\"-> TOOL: Fetching last {lines} log lines for {server_id}...\")\n\n",
        "    log_database = {\n",
        "        \"payment-server-01\": [\n",
        "            \"[INFO] Request received /pay/v1\",\n",
        "            \"[WARN] CPU threshold exceeded 90%\",\n",
        "            \"[WARN] Thread pool exhaustion\",\n",
        "            \"[CRITICAL] Process hung, not accepting new connections\",\n",
        "            \"[ERROR] Timeout waiting for thread\"\n",
        "        ],\n",
        "        \"db-node-02\": [\n",
        "            \"[INFO] Backup started\",\n",
        "            \"[INFO] Backup completed successfully\",\n",
        "            \"[INFO] User query executed in 12ms\",\n",
        "            \"[INFO] Health check: OK\",\n",
        "            \"[INFO] Replication sync active\"\n",
        "        ],\n",
        "        \"auth-service-03\": [\n",
        "            \"[INFO] Token validated user_882\",\n",
        "            \"[WARN] Garbage collection taking too long (>5s)\",\n",
        "            \"[ERROR] java.lang.OutOfMemoryError: Java heap space\",\n",
        "            \"[CRITICAL] Application crashing due to memory leak\",\n",
        "            \"[INFO] Restarting context...\"\n",
        "        ],\n",
        "        \"search-index-09\": [\n",
        "            \"[INFO] Indexing started\",\n",
        "            \"[ERROR] Connection refused: elastic-cluster-main:9200\",\n",
        "            \"[ERROR] Failed to write document ID 4432\",\n",
        "            \"[CRITICAL] Dependency Unreachable: Search Engine is down\",\n",
        "            \"[ERROR] Retrying in 30s...\"\n",
        "        ],\n",
        "        \"frontend-node-04\": [\n",
        "            \"[INFO] GET /home 200 OK\",\n",
        "            \"[INFO] GET /assets/logo.png 200 OK\",\n",
        "            \"[INFO] GET /login 200 OK\",\n",
        "            \"[INFO] GET /api/v1/status 200 OK\",\n",
        "            \"[INFO] Health check passed\"\n",
        "        ]\n",
        "    }\n\n",
        "    default_logs = [\"[INFO] System stable\", \"[INFO] Heartbeat signal received\"]\n\n",
        "    logs = log_database.get(server_id, default_logs)\n",
        "    return json.dumps({\"logs\": logs[:lines]})\n\n",
        "# --- Tool 3: Restart Service ---\n",
        "def restart_service(server_id: str) -> str:\n",
        "    \"\"\"\n",
        "    Agent will restart the server when CPU hits 98% and Memory Usage hits 40%\n",
        "    1. Print a message saying \"-> TOOL: Restarting service...\"\n",
        "    2. Return a JSON string confirming the restart was successful.\n",
        "       Example return: '{\"status\": \"success\", \"message\": \"Server restarted successfully\"}'\n",
        "    \"\"\"\n",
        "    print(f\"-> TOOL: RESTARTING SERVICE {server_id}...\")\n",
        "    return json.dumps({\"status\": \"success\", \"message\": f\"Server {server_id} restarted successfully\"})\n\n",
        "# --- Tool 4: Escalation Tool ---\n",
        "def escalate_to_engineer(summary: str) -> str:\n",
        "    \"\"\"\n",
        "    Agent will escalate the complex issue to Engineers/Or Escalation Team\n",
        "    1. Print a message saying \"-> TOOL: Escalating to human...\"\n",
        "    2. Return a JSON string confirming the ticket was created.\n",
        "    \"\"\"\n",
        "    ticket_id = f\"INC-{random.randint(1000, 9999)}\"\n",
        "    print(f\"-> TOOL: Escalating to human for: {summary}. Generating Ticket ID: {ticket_id}...\")\n",
        "    return json.dumps({\"status\": \"success\", \"message\": f\"Escalation ticket created for: {summary}\", \"ticket_id\": ticket_id})\n\n",
        "# Map functions for the agent execution loop\n",
        "AVAILABLE_FUNCTIONS = {\n",
        "    \"get_server_health\": get_server_health,\n",
        "    \"fetch_recent_logs\": fetch_recent_logs,\n",
        "    \"restart_service\": restart_service,\n",
        "    \"escalate_to_engineer\": escalate_to_engineer,\n",
        "}\n\n",
        "# Define Agent Schema\n",
        "tools_schema = [\n",
        "    {\n",
        "        \"type\": \"function\",\n",
        "        \"function\": {\n",
        "            \"name\": \"get_server_health\",\n",
        "            \"description\": \"Checks the current CPU and memory usage of a specific server.\",\n",
        "            \"parameters\": {\n",
        "                \"type\": \"object\",\n",
        "                \"properties\": {\n",
        "                    \"server_id\": {\"type\": \"string\", \"description\": \"The ID of the server, e.g., 'payment-server-01'\"}\n",
        "                },\n",
        "                \"required\": [\"server_id\"]\n",
        "            }\n",
        "        }\n",
        "    },\n",
        "    {\n",
        "        \"type\": \"function\",\n",
        "        \"function\": {\n",
        "            \"name\": \"fetch_recent_logs\",\n",
        "            \"description\": \"Retrieves the most recent log entries from a server to diagnose errors.\",\n",
        "            \"parameters\": {\n",
        "                \"type\": \"object\",\n",
        "                \"properties\": {\n",
        "                    \"server_id\": {\"type\": \"string\", \"description\": \"The ID of the server.\"},\n",
        "                    \"lines\": {\"type\": \"integer\", \"description\": \"Number of log lines to fetch.\"}\n",
        "                },\n",
        "                \"required\": [\"server_id\"]\n",
        "            }\n",
        "        }\n",
        "    },\n",
        "    {\n",
        "        \"type\": \"function\",\n",
        "        \"function\": {\n",
        "            \"name\": \"restart_service\",\n",
        "            \"description\": \"Restarts a specified server to resolve performance or stability issues. Used when CPU or memory usage is critically high.\",\n",
        "            \"parameters\": {\n",
        "                \"type\": \"object\",\n",
        "                \"properties\": {\n",
        "                    \"server_id\": {\"type\": \"string\", \"description\": \"The ID of the server to be restarted.\"}\n",
        "                },\n",
        "                \"required\": [\"server_id\"]\n",
        "            }\n",
        "        }\n",
        "    },\n",
        "    {\n",
        "        \"type\": \"function\",\n",
        "        \"function\": {\n",
        "            \"name\": \"escalate_to_engineer\",\n",
        "            \"description\": \"Escalates the issue to a human engineer when automated fixes fail or the error is unknown or complex, providing a summary of the problem.\",\n",
        "            \"parameters\": {\n",
        "                \"type\": \"object\",\n",
        "                \"properties\": {\n",
        "                     \"summary\": {\"type\": \"string\", \"description\": \"A concise summary of the issue requiring escalation.\"}\n",
        "                },\n",
        "                \"required\": [\"summary\"]\n",
        "            }\n",
        "        }\n",
        "    }\n",
        "]\n\n",
        "# The Agent Execution Loop\n",
        "def run_it_agent(user_issue: str):\n",
        "    print(f\"\\n--- New Incident: {user_issue} ---\")\n\n",
        "    messages = [\n",
        "        {\"role\": \"system\", \"content\": \"You are a Level 1 IT Responder. Investigate server issues. \"\n",
        "                                      \"If CPU or Memory is > 90%, restart the service. If logs show critical dependency errors (like connection refused) that a restart won't fix, escalate to an engineer.\"},\n",
        "        {\"role\": \"user\", \"content\": user_issue}\n",
        "    ]\n\n",
        "    while True:\n",
        "        print(\"\\n[AI Thinking...]\")\n",
        "        response = client.chat.completions.create(\n",
        "            model=\"gpt-4o\", # Model changed to gpt-4o as per previous instructions to resolve access error\n",
        "            messages=messages,\n",
        "            tools=tools_schema,\n",
        "            tool_choice=\"auto\"\n",
        "        )\n\n",
        "        response_msg = response.choices[0].message\n",
        "        messages.append(response_msg)\n\n",
        "        if response_msg.tool_calls:\n",
        "            for tool_call in response_msg.tool_calls:\n",
        "                func_name = tool_call.function.name\n",
        "                func_args = json.loads(tool_call.function.arguments)\n\n",
        "                # Retrieve the actual python function based on name\n",
        "                function_to_call = AVAILABLE_FUNCTIONS.get(func_name)\n\n",
        "                if function_to_call:\n",
        "                    # Execute the function\n",
        "                    tool_output = function_to_call(**func_args)\n\n",
        "                    # Append the result to messages\n",
        "                    messages.append({\n",
        "                        \"role\": \"tool\",\n",
        "                        \"tool_call_id\": tool_call.id,\n",
        "                        \"name\": func_name,\n",
        "                        \"content\": tool_output\n",
        "                    })\n\n",
        "        else:\n",
        "            print(f\"\\n[FINAL RESPONSE]: {response_msg.content}\")\n",
        "            break"
      ],
      "metadata": {
        "id": "feec5145"
      },
      "outputs": [],
      "execution_count": null
    },
    {
      "cell_type": "markdown",
      "source": [
        "You can now run this cell to ensure all the agent components are loaded. Then you can test the agent with the scenarios below."
      ],
      "metadata": {
        "id": "9d041658"
      }
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Example Test Scenarios\n\n",
        "Run these cells one by one to see the agent in action."
      ],
      "metadata": {
        "id": "53f2555e"
      }
    },
    {
      "cell_type": "code",
      "source": [
        "print("\\n" + "="*50 + "\\n")\n",
        "run_it_agent(\"The payment-server-01 is extremely slow and timing out.\")"
      ],
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "id": "288be28e",
        "outputId": "f989c922-0a1e-4504-f5fe-cf099d8b7a60"
      },
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            "==================================================\\n",
